In [ ]:
import random
import re
from typing import List, Tuple, Callable, Dict

import numpy as np
from transformers import pipeline, set_seed

In [ ]:
set_seed(42)
random.seed(42)
np.random.seed(42)

In [ ]:
clf = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    device=-1,  # CPU
)

Device set to use cpu


In [ ]:
def predict_all_scores(texts: List[str]) -> List[Dict[str, float]]:
    """
    Возвращает для каждого текста dict {'POSITIVE': p_pos, 'NEGATIVE': p_neg}
    """
    outputs = clf(texts, return_all_scores=True, truncation=True)
    results = []
    for item in outputs:
        d = {x["label"]: float(x["score"]) for x in item}
        results.append(d)
    return results

In [ ]:
def predict_label(text: str) -> str:
    scores = predict_all_scores([text])[0]
    return "POSITIVE" if scores["POSITIVE"] >= scores["NEGATIVE"] else "NEGATIVE"

In [ ]:
MFT_POS = [
    "I absolutely loved this movie. It was brilliant.",
    "This is fantastic work!",
    "The product quality is great and I recommend it.",
    "What an amazing experience.",
    "The dinner was delicious and the service was excellent.",
    "I am very happy with this purchase.",
    "A wonderful story with excellent acting.",
    "The upgrade made everything better.",
    "I enjoyed every moment of it.",
    "Highly satisfying result overall."
]

MFT_NEG = [
    "I hated this movie. It was awful.",
    "This is terrible and disappointing.",
    "The product is broken and useless.",
    "What a horrible experience.",
    "The food was cold and the waiter was rude.",
    "I regret buying this.",
    "A boring story with poor acting.",
    "The update made things worse.",
    "I disliked almost everything about it.",
    "A very unsatisfying result overall."
]

### Утилиты для Perturbation и Invariance тестов

In [ ]:
WORD_RE = re.compile(r"\b\w+\b", re.UNICODE)

In [ ]:
def swap_adjacent_chars(word: str) -> str:
    if len(word) < 4:
        return word
    i = random.randint(1, len(word) - 2)  # не трогаем крайние
    lst = list(word)
    lst[i], lst[i + 1] = lst[i + 1], lst[i]
    return "".join(lst)

In [ ]:
def random_typos(text: str, typo_prob: float = 0.2) -> str:
    def repl(m):
        w = m.group(0)
        return swap_adjacent_chars(w) if random.random() < typo_prob else w
    return WORD_RE.sub(repl, text)

In [ ]:
def random_case(text: str, prob: float = 0.2) -> str:
    chars = []
    for ch in text:
        if ch.isalpha() and random.random() < prob:
            ch = ch.upper() if ch.islower() else ch.lower()
        chars.append(ch)
    return "".join(chars)

In [ ]:
def duplicate_spaces(text: str) -> str:
    # нормализуем, потом удвоим пробелы
    compact = re.sub(r"\s+", " ", text.strip())
    return compact.replace(" ", "  ")

In [ ]:
def add_punctuation(text: str) -> str:
    return text + "!!!"

def add_filler(text: str) -> str:
    return "In my opinion, " + text

SYNONYM_MAP = {
    r"\bmovie\b": "film",
    r"\bmovies\b": "films",
    r"\bgreat\b": "excellent",
    r"\bterrible\b": "horrible",
    r"\bgood\b": "nice",
    r"\bbad\b": "poor",
    r"\bproduct\b": "item",
    r"\bservice\b": "assistance",
}
def replace_synonyms(text: str) -> str:
    out = text
    for pat, repl in SYNONYM_MAP.items():
        out = re.sub(pat, repl, out, flags=re.IGNORECASE)
    return out

### Minimal Functionality Test

In [ ]:
def test_mft(samples: List[Tuple[str, str]]) -> None:
    print("MFT: smoke-тесты")
    texts = [x for x, _ in samples]
    expected = [y for _, y in samples]
    preds = [predict_label(t) for t in texts]
    passed = sum(int(p == e) for p, e in zip(preds, expected))
    total = len(samples)
    print(f"Passed {passed}/{total}")

    if passed < total:
        print("Failed cases:")
        for i, (t, e, p) in enumerate(zip(texts, expected, preds)):
            if p != e:
                print(f"  [{i}] expected={e}, got={p} | {t}")

### Perturbation Test: незначительный шум не должен радикально менять уверенность/метку

In [ ]:
def test_perturbation(
    texts: List[str],
    noise_fn: Callable[[str], str],
    max_delta: float = 0.20,
    require_same_label: bool = True
) -> None:
    print("Perturbation Test: устойчивость к шуму")
    base_scores = predict_all_scores(texts)
    noisy_texts = [noise_fn(t) for t in texts]
    noisy_scores = predict_all_scores(noisy_texts)

    fails = 0
    for i, (t, b, t2, n) in enumerate(zip(texts, base_scores, noisy_texts, noisy_scores)):
        base_label = "POSITIVE" if b["POSITIVE"] >= b["NEGATIVE"] else "NEGATIVE"
        noisy_label = "POSITIVE" if n["POSITIVE"] >= n["NEGATIVE"] else "NEGATIVE"
        delta = abs(n["POSITIVE"] - b["POSITIVE"])
        ok_label = (base_label == noisy_label) if require_same_label else True
        ok_delta = (delta <= max_delta)

        if not (ok_label and ok_delta):
            fails += 1
            print(f"- Fail [{i}]:")
            print(f"  base:  {t}")
            print(f"  noisy: {t2}")
            print(f"  base_label={base_label} p_pos={b['POSITIVE']:.3f}, noisy_label={noisy_label} p_pos={n['POSITIVE']:.3f}, Δ={delta:.3f}")

    print(f"Passed {len(texts) - fails}/{len(texts)}")

### Invariance Test: преобразования, не меняющие смысл, не должны менять класс

In [ ]:
def test_invariance(texts: List[str], transforms: List[Callable[[str], str]]) -> None:
    print("Invariance Test: неизменность класса при эквивалентных преобразованиях")
    base_labels = [predict_label(t) for t in texts]

    total = 0
    fails = 0
    for tf in transforms:
        transformed = [tf(t) for t in texts]
        labels = [predict_label(t) for t in transformed]
        for i, (bl, l) in enumerate(zip(base_labels, labels)):
            total += 1
            if bl != l:
                fails += 1
                print(f"- Fail [idx={i}, tf={tf.__name__}] {texts[i]}")
    print(f"Passed {total - fails}/{total}")

### Directional Expectations Test: изменения с ожидаемым направлением

In [ ]:
def test_directional(
    pairs: List[Tuple[str, str, str]],
    margin: float = 0.10
) -> None:
    """
    pairs: (src, dst, expectation)
      expectation in {
        'more_positive', 'less_positive',
        'flip_to_negative', 'flip_to_positive'
      }
    """
    print("Directional Expectations Test: ожидаемое направление изменения")
    src_texts = [p[0] for p in pairs]
    dst_texts = [p[1] for p in pairs]
    src_scores = predict_all_scores(src_texts)
    dst_scores = predict_all_scores(dst_texts)

    def label_from(scores): return "POSITIVE" if scores["POSITIVE"] >= scores["NEGATIVE"] else "NEGATIVE"

    fails = 0
    for i, ((s, d, exp), sc_s, sc_d) in enumerate(zip(pairs, src_scores, dst_scores)):
        label_s = label_from(sc_s)
        label_d = label_from(sc_d)
        ppos_s, ppos_d = sc_s["POSITIVE"], sc_d["POSITIVE"]

        ok = False
        if exp == "more_positive":
            ok = (ppos_d >= ppos_s + margin)
        elif exp == "less_positive":
            ok = (ppos_d <= ppos_s - margin)
        elif exp == "flip_to_negative":
            ok = (label_s == "POSITIVE" and label_d == "NEGATIVE")
        elif exp == "flip_to_positive":
            ok = (label_s == "NEGATIVE" and label_d == "POSITIVE")
        else:
            raise ValueError(f"Unknown expectation: {exp}")

        if not ok:
            fails += 1
            print(f"- Fail [{i}] exp={exp}")
            print(f"  src: {s}")
            print(f"  dst: {d}")
            print(f"  src: label={label_s} p_pos={ppos_s:.3f}")
            print(f"  dst: label={label_d} p_pos={ppos_d:.3f}")

    print(f"Passed {len(pairs) - fails}/{len(pairs)}")

Запускаем все тесты

In [ ]:
mft_samples = [(x, "POSITIVE") for x in MFT_POS] + [(x, "NEGATIVE") for x in MFT_NEG]
test_mft(mft_samples)

MFT: smoke-тесты
Passed 20/20


In [ ]:
subset = MFT_POS[:5] + MFT_NEG[:5]
print("\n-- Perturbation: random_typos --")
test_perturbation(subset, noise_fn=lambda t: random_typos(t, 0.25), max_delta=0.20, require_same_label=True)

print("\n-- Perturbation: random_case --")
test_perturbation(subset, noise_fn=lambda t: random_case(t, 0.3), max_delta=0.20, require_same_label=True)


-- Perturbation: random_typos --
Perturbation Test: устойчивость к шуму
Passed 10/10

-- Perturbation: random_case --
Perturbation Test: устойчивость к шуму
Passed 10/10


In [ ]:
inv_texts = [
    "This movie is good.",
    "The service was excellent.",
    "I dislike this product.",
    "The experience was terrible."
]
transforms = [add_punctuation, add_filler, duplicate_spaces, replace_synonyms]
print("\n-- Invariance --")
test_invariance(inv_texts, transforms)


-- Invariance --
Invariance Test: неизменность класса при эквивалентных преобразованиях
Passed 16/16


In [ ]:
dir_pairs = [
    # усиление позитивного
    ("The movie is good.", "The movie is very good.", "more_positive"),
    ("I like this product.", "I really like this product.", "more_positive"),

    # усиление негативного
    ("The movie is bad.", "The movie is extremely bad.", "less_positive"),
    ("The service was poor.", "The service was very poor.", "less_positive"),

    # отрицание — ожидание ухудшения
    ("The service was good.", "The service was not good.", "less_positive"),
    ("This is nice.", "This is not nice.", "less_positive"),

    # контраст «но» — ожидание ухудшения
    ("The plot is okay.", "The plot is okay, but the acting is terrible.", "less_positive"),

    # Явный переворот класса
    ("I loved this.", "I did not love this.", "flip_to_negative"),
    ("I hate this.", "I do not hate this.", "flip_to_positive"),
]
print("\n-- Directional --")
test_directional(dir_pairs, margin=0.10)


-- Directional --
Directional Expectations Test: ожидаемое направление изменения
- Fail [0] exp=more_positive
  src: The movie is good.
  dst: The movie is very good.
  src: label=POSITIVE p_pos=1.000
  dst: label=POSITIVE p_pos=1.000
- Fail [1] exp=more_positive
  src: I like this product.
  dst: I really like this product.
  src: label=POSITIVE p_pos=1.000
  dst: label=POSITIVE p_pos=1.000
- Fail [2] exp=less_positive
  src: The movie is bad.
  dst: The movie is extremely bad.
  src: label=NEGATIVE p_pos=0.000
  dst: label=NEGATIVE p_pos=0.000
- Fail [3] exp=less_positive
  src: The service was poor.
  dst: The service was very poor.
  src: label=NEGATIVE p_pos=0.000
  dst: label=NEGATIVE p_pos=0.000
Passed 5/9
